# 3주차 — SELECT ① 조회의 기본기
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다.
어떤 셀이 오류로 막히거나 데이터가 엉켰다면, 아무 셀에서 `reset_db()`를 실행하면 처음 상태로 돌아갑니다.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, shutil, sqlite3, urllib.request
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"
DB_PATH = "mini_emr.sqlite"
WORK_PATH = DB_PATH   # 실습에 사용할 파일

con = None

def reset_db():
    """DB를 원본 상태로 되돌리고 다시 연결 (막히면 언제든 호출)"""
    global con
    if con is not None:
        con.close()
    try:
        urllib.request.urlretrieve(DB_URL, DB_PATH)
    except Exception as e:
        if not os.path.exists(DB_PATH):
            raise RuntimeError(
                "DB 다운로드 실패 — 왼쪽 파일 탭에 mini_emr.sqlite를 업로드한 뒤 이 셀을 다시 실행하세요."
            ) from e
        print("다운로드 실패 — 업로드되어 있는 파일을 사용합니다.")
    if WORK_PATH != DB_PATH:
        shutil.copyfile(DB_PATH, WORK_PATH)   # 원본 보존, 실습은 사본에서
    con = sqlite3.connect(WORK_PATH)
    con.execute("PRAGMA foreign_keys = ON")   # SQLite는 기본으로 꺼져 있음 — 반드시 켠다
    assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1

def q(sql):
    """조회(SELECT) 실행 → 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """데이터/구조를 바꾸는 SQL(INSERT/UPDATE/DELETE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

reset_db()
q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. SELECT — 전부, 혹은 골라서

In [ ]:
# 환자 테이블 전체 (LIMIT으로 자르기)
q("""
SELECT * FROM patient LIMIT 5
""")

In [ ]:
# 특정 컬럼만
q("""
SELECT patient_id, birth_date FROM patient LIMIT 5
""")

In [ ]:
# 별칭(AS)
q("""
SELECT patient_id AS 환자번호, blood_type AS 혈액형 FROM patient LIMIT 5
""")

## 2. DISTINCT — 값의 종류 탐색

In [ ]:
# 우리 병원의 진료과 목록
q("""
SELECT DISTINCT department FROM visit ORDER BY 1
""")

In [ ]:
# 의료진 직종 목록
q("""
SELECT DISTINCT role FROM staff
""")

## 3. ORDER BY · LIMIT

In [ ]:
# 나이 많은 순 (생년월일 오름차순)
q("""
SELECT patient_id, birth_date, sex FROM patient ORDER BY birth_date ASC LIMIT 10
""")

In [ ]:
# 가장 최근 내원 5건
q("""
SELECT visit_id, patient_id, visit_date, department
FROM visit ORDER BY visit_date DESC LIMIT 5
""")

## 연습문제 — 먼저 직접 풀어보세요

In [ ]:
# [연습 Q1] 검사(lab) 테이블에는 어떤 검사 종류들이 있는가?
# 힌트: DISTINCT
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 처방에서 약품명과 용량만, 약품명 가나다순으로
# 힌트: ORDER BY
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 가장 어린 환자 3명의 ID와 생년월일은?
# 힌트: birth_date DESC + LIMIT
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] 영상(imaging) 테이블의 촬영종류 목록은?
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 검사 종류**
```sql
SELECT DISTINCT test_name FROM lab ORDER BY 1;
```

**A2. 약품명·용량 가나다순**
```sql
SELECT drug_name, dose FROM prescription ORDER BY drug_name;
```

**A3. 가장 어린 환자 3명**
```sql
SELECT patient_id, birth_date FROM patient ORDER BY birth_date DESC LIMIT 3;
```

**A4. 촬영종류 목록**
```sql
SELECT DISTINCT modality FROM imaging;
```